## Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import json

## Get Time Series

In [ ]:
folder_path = '/Users/fiarresga/Desktop/2025/Thesis/Biobank/Predictor/sts_all'

In [ ]:
columns = ['filename', 'time_series', 'Mean', 'STD', 'skewR', 'kurtR', 'perc95', 
           'perc5', 'dPerc', 'Dominant frequency', 'Peak power', 'PPFd', 
           'Entropy', 'PF_sum', 'P_sum']

rows = []

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, header=None)
        df = df.drop(0)
        
        imputed_series = pd.to_numeric(df[0].fillna(method='ffill'), errors='coerce').dropna()
        imputed_array = imputed_series.to_numpy()

        rows.append([filename[:-14], imputed_array] + [np.nan] * (len(columns) - 2))


all_time_series = pd.DataFrame(rows, columns=columns)
all_time_series

In [ ]:
"""all_time_series = pd.DataFrame(columns=['filename', 'time_series', 'Mean', 'STD', 'skewR', 'kurtR', 'perc95', 'perc5', 'dPerc', 'Dominant frequency', 'Peak power', 'PPFd', 'Entropy', 'PF_sum', 'P_sum'])

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, header=None)
        df = df.drop(0)
        imputed_series = df[[0]].fillna(method='ffill')
        imputed_series = pd.to_numeric(imputed_series.squeeze(), errors='coerce').dropna()
        imputed_array = imputed_series.to_numpy()


        new_row = pd.DataFrame([[filename] + [imputed_array] + [np.nan] * (len(all_time_series.columns) - 2)], columns=all_time_series.columns)
        all_time_series = pd.concat([all_time_series, new_row], ignore_index=True)

all_time_series['filename'] = all_time_series['filename'].str.slice(0, -14)

all_time_series"""

## Get Predictors Time Domain

In [ ]:
def calculate_mpd(time_series):
    mean_value = np.mean(time_series)
    temp = time_series - mean_value
    temp = (temp**2)/(len(time_series))
    temp = np.sqrt(temp)
    return np.round(temp,5)

all_time_series['MPD'] = all_time_series['time_series'].apply(calculate_mpd)
#all_time_series

In [ ]:
def calulate_perc95(time_series):
    percentile_95 = np.percentile(time_series, 95)
    filtered_data = time_series[time_series >= percentile_95]
    mean_95th_percentile = np.mean(filtered_data)
    return mean_95th_percentile

def calulate_perc5(time_series):
    percentile_95 = np.percentile(time_series, 5)
    filtered_data = time_series[time_series <= percentile_95]
    mean_95th_percentile = np.mean(filtered_data)
    return mean_95th_percentile

all_time_series['perc95'] = all_time_series['time_series'].apply(calulate_perc95)
all_time_series['perc5'] = all_time_series['time_series'].apply(calulate_perc5)
all_time_series['dPerc'] = all_time_series['perc95'] - all_time_series['perc5']
#all_time_series

In [ ]:
def calculate_mad(time_series):
    mean_value = np.mean(time_series)
    temp = np.abs(time_series - mean_value) * (1/(len(time_series)))
    return np.round(temp, 5)

all_time_series['MAD'] = all_time_series['time_series'].apply(calculate_mad)
#all_time_series

In [ ]:
def calculate_skewR(time_series):
    mean_x = np.mean(time_series)
    std_x = np.std(time_series)
    sum_x_minus_mean = np.sum(time_series - mean_x)
    term = (sum_x_minus_mean / std_x) ** 3
    n = len(time_series)
    scaling_factor = (n / (n - 1)) * (n / (n - 2))
    result = scaling_factor * term
    return result

all_time_series['skewR'] = all_time_series['time_series'].apply(calculate_skewR)
#all_time_series

In [ ]:
def calculate_kurtR(time_series):
    mean_x = np.mean(time_series)
    std_x = np.std(time_series)
    n = len(time_series)
    sum_x_minus_mean = np.sum(time_series - mean_x)
    term = (sum_x_minus_mean / std_x) ** 4 - ((3*(n-1)**2)/((n-2)*(n-3)))
    scaling_factor = ((n*(n+1)) / (n - 1)) * (n*(n+1) / (n - 2)) * (n*(n+1) / (n - 3))
    result = scaling_factor * term 
    return result
    
all_time_series['kurtR'] = all_time_series['time_series'].apply(calculate_kurtR)
#all_time_series

In [ ]:
from scipy.signal import find_peaks

def calculate_steps(time_series):
    window_size = 5
    resultant_acc_smoothed = np.convolve(time_series, np.ones(window_size)/window_size, mode='same')
    peaks, _ = find_peaks(resultant_acc_smoothed, height=3, distance=20)
    return len(peaks)


all_time_series['Steps'] = all_time_series['time_series'].apply(calculate_steps)

In [ ]:
all_time_series['Mean'] = all_time_series['time_series'].apply(np.mean)
all_time_series['STD'] = all_time_series['time_series'].apply(np.std)
#all_time_series

In [ ]:
all_time_series['MAD'] = all_time_series['MAD'].apply(np.std)
all_time_series['MPD'] = all_time_series['MPD'].apply(np.std)


## Get Predictors Frequency Domain

In [ ]:
def calculate_fft_probabilities(signal):
    sampling_frequency = 100
    mean_acceleration = np.mean(signal)

    #Detrend
    signal = signal - mean_acceleration

    std_acceleration = np.std(signal)
    if std_acceleration > 0:
        signal = signal / std_acceleration
    else:
        signal = signal

    n = len(signal)
    fft_values = np.fft.fft(signal)
    frequencies = np.fft.fftfreq(n, d=1/sampling_frequency)


    power_spectrum = np.abs(fft_values[:n//2])**2
    frequencies = frequencies[:n//2]

    total_power = np.sum(power_spectrum)

    probabilities = power_spectrum / total_power

    dominant_index = np.argmax(probabilities)
    dominant_frequency = frequencies[dominant_index]
    power_at_dominant = power_spectrum[dominant_index]

    product_peak_power_and_frequency = dominant_frequency * power_at_dominant

    probabilities_safe = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities_safe * np.log2(probabilities_safe))

    sum_freq_times_power = np.sum(frequencies * power_spectrum)

    return frequencies, probabilities, dominant_frequency, power_at_dominant, product_peak_power_and_frequency, entropy, sum_freq_times_power, total_power

In [ ]:
all_time_series['Dominant frequency'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[2])
all_time_series['Peak power'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[3])
all_time_series['PPFd'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[4])
all_time_series['Entropy'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[5])
all_time_series['PF_sum'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[6])
all_time_series['P_sum'] = all_time_series['time_series'].apply(lambda x: calculate_fft_probabilities(x)[7])
all_time_series.head()

## Get Predictors Stats

In [ ]:
data_participants = pd.read_csv('data_participant.csv')

data_participants['eid'] = data_participants['eid'].astype('object')

data_participants['Diastolic'] = data_participants['p4079_i0_a0'].combine_first(data_participants['p94_i0_a0'])
data_participants['Systolic'] = data_participants['p4080_i0_a0'].combine_first(data_participants['p93_i0_a0'])
data_participants.head()

In [ ]:
all_time_series['filename'] = all_time_series['filename'].astype(str)
data_participants['eid'] = data_participants['eid'].astype(str)

In [ ]:
all_time_series = pd.merge(all_time_series, data_participants, how='left', left_on='filename', right_on='eid')

In [ ]:
all_time_series.to_csv('whole_file_for_recommender.csv')

## Get Target

In [ ]:
def get_file_info(partial_name, folder_path):
    for filename in os.listdir(folder_path):
        if filename.startswith(partial_name) and filename.endswith('.json'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as f:
                data = json.load(f)
                mean_times_light = data.get('light-overall-avg', None)
                std_times_light = data.get('light-overall-sd', None)

                mean_times_moderate = data.get('moderate-vigorous-overall-avg', None)
                std_times_moderate = data.get('moderate-vigorous-overall-sd', None)

                mean_times_sed = data.get('sedentary-overall-avg', None)
                std_times_sed = data.get('sedentary-overall-sd', None)

                mean_times_sleep = data.get('sleep-overall-avg', None)
                std_times_sleep = data.get('sleep-overall-sd', None)

                return mean_times_light, std_times_light, mean_times_moderate, std_times_moderate, mean_times_sed, std_times_sed, mean_times_sleep, std_times_sleep
    return None, None, None, None, None, None, None, None 


In [ ]:
mean_light_list = []
std_light_list = []

mean_moderate_list = []
std_moderate_list = []

mean_sed_list = []
std_sed_list = []

mean_sleep_list = []
std_sleep_list = []

folder_path_summary = '/Users/fiarresga/Desktop/2025/Thesis/Biobank/Predictor/Calibrated_Data'


for index, row in all_time_series.iterrows():
    partial_name = row['filename']
    mean_times_light, std_times_light, mean_times_moderate, std_times_moderate, mean_times_sed, std_times_sed, mean_times_sleep, std_times_sleep = get_file_info(partial_name, folder_path_summary)
    
    mean_light_list.append(mean_times_light)
    std_light_list.append(std_times_light)

    mean_moderate_list.append(mean_times_moderate)
    std_moderate_list.append(std_times_moderate)

    mean_sed_list.append(mean_times_sed)
    std_sed_list.append(std_times_sed)

    mean_sleep_list.append(mean_times_sleep)
    std_sleep_list.append(std_times_sleep)


all_time_series['mean_light'] = mean_light_list
all_time_series['std_light'] = std_light_list

all_time_series['mean_moderate'] = mean_moderate_list
all_time_series['std_moderate'] = std_moderate_list

all_time_series['mean_sed'] = mean_sed_list
all_time_series['std_sed'] = std_sed_list

all_time_series['mean_sleep'] = mean_sleep_list
all_time_series['std_sleep'] = std_sleep_list

all_time_series

In [ ]:
def classify_activity(row): 
    vigorous_mean = row['mean_moderate']
    light_mean = row['mean_light']
    sedentary_mean = row['mean_sed']


    MODERATE_VIGOROUS = 0.03
    HIGH_LIGHT = 0.25
    
    if vigorous_mean >= MODERATE_VIGOROUS:
        return "Active"
    if light_mean >= HIGH_LIGHT:
        return "Active"
    

    
    return 'Sedentary'

In [ ]:
all_time_series['activity_class'] = all_time_series.apply(classify_activity, axis=1)


In [ ]:
all_time_series['activity_class'].value_counts()

In [ ]:
all_time_series.to_csv('predictors_plus_target_calibrated.csv')